In [ ]:
# Cell 1 — Install PaddleOCR + GPU support (corrected for Kaggle T4 x2)

# ---- Step 1: Detect the actual CUDA version in this Kaggle image ----
!nvcc --version
!nvidia-smi | head -20

import subprocess, re

def get_cuda_version():
    try:
        out = subprocess.check_output(["nvcc", "--version"], text=True)
        m = re.search(r"release (\d+)\.(\d+)", out)
        if m:
            return f"cu{m.group(1)}{m.group(2)}"
    except Exception:
        pass
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
            text=True,
        )
        print("Driver:", out.strip())
    except Exception:
        pass
    return None

CUDA_TAG = get_cuda_version()
print("Detected CUDA tag:", CUDA_TAG)

# ---- Step 2: Pick the matching PaddlePaddle GPU wheel ----
# PaddlePaddle 3.x stable wheels are published for cu118, cu126, and cu129.
# Map detected version → available index.
PADDLE_INDEX = {
    "cu118": "https://www.paddlepaddle.org.cn/packages/stable/cu118/",
    "cu126": "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    "cu129": "https://www.paddlepaddle.org.cn/packages/stable/cu129/",
}

# If detection failed, default to cu118 (most common fallback) — but check the
# printed nvcc output above and adjust manually if needed.
idx_url = PADDLE_INDEX.get(CUDA_TAG, PADDLE_INDEX["cu118"])
print(f"Using PaddlePaddle index: {idx_url}")

# ---- Step 3: Install PaddlePaddle GPU ----
# Use --index-url (not -i) so this index is explicitly scoped to this command
# and cannot be silently overridden by later pip calls.
!pip install -q paddlepaddle-gpu==3.0.0 --index-url {idx_url} --extra-index-url https://pypi.org/simple

# ---- Step 4: Install PaddleOCR (core first, then doc-parser separately) ----
!pip install -q paddleocr pymupdf pillow opencv-python-headless

# doc-parser pulls PP-StructureV3 etc. Install it separately so a failure here
# does not roll back the core paddleocr install.
!pip install -q "paddleocr[doc-parser]"

# ---- Step 5: Verify ----
import paddle

print("\n=== Verification ===")
print("Paddle version:", paddle.__version__)
print("CUDA compiled:", paddle.is_compiled_with_cuda())
print("GPU count:", paddle.device.cuda.device_count())

if paddle.device.cuda.device_count() > 0:
    print("CUDA version:", paddle.version.cuda())
    for i in range(paddle.device.cuda.device_count()):
        print(f"  GPU {i}: {paddle.device.cuda.get_device_name(i)}")
    paddle.set_device("gpu:0")
    print("Default device set to: gpu:0")
else:
    print("⚠️ No GPU detected. Enable GPU accelerator in Kaggle notebook settings.")
    print("   Settings → Accelerator → GPU T4 x2")

In [1]:
# Cell 2 — Imports, paths, config, inventory
# Requires fresh kernel. Do NOT re-run without restarting.

import importlib.util, os, sys

# ---- Guarded one-time patch: block modelscope from importing torch ----
if not getattr(importlib.util.find_spec, "_paddle_kaggle_patched", False):
    _real_find_spec = importlib.util.find_spec  # captured in closure, not global

    def _patched_find_spec(name, *args, **kwargs):
        if name == "torch":
            return None
        return _real_find_spec(name, *args, **kwargs)

    _patched_find_spec._paddle_kaggle_patched = True
    importlib.util.find_spec = _patched_find_spec

# ---- Skip slow model-source connectivity check ----
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

# ---- Normal imports ----
import io, json, time, shutil
from pathlib import Path

import numpy as np
import pymupdf
from PIL import Image

import paddle
import paddleocr
from paddleocr import PaddleOCR

print("PaddlePaddle :", paddle.__version__)
print("PaddleOCR    :", paddleocr.__version__)
print("GPUs         :", paddle.device.cuda.device_count())
print("CUDA built   :", paddle.is_compiled_with_cuda())

PADDLEOCR_MAJOR = int(paddleocr.__version__.split(".")[0])
print(f"→ Will use PaddleOCR {PADDLEOCR_MAJOR}.x API")

# ---- Locate input dataset (recursive) ----
SOURCES = ["alerts", "fdc", "gazette", "public_notices"]

def find_input_root(source_names, search_root=Path("/kaggle/input")) -> Path:
    if not search_root.is_dir():
        raise RuntimeError(f"{search_root} not found — not running on Kaggle?")
    expected = set(source_names)
    candidates = []
    for dirpath, dirnames, _ in os.walk(search_root):
        overlap = len(set(dirnames) & expected)
        if overlap:
            candidates.append((overlap, Path(dirpath)))
    if not candidates:
        raise RuntimeError(f"None of {source_names} found under {search_root}.")
    candidates.sort(key=lambda x: (-x[0], len(x[1].parts)))
    best_overlap, best_path = candidates[0]
    print(f"  → matched {best_overlap}/{len(expected)} source dirs at {best_path}")
    return best_path

INPUT_ROOT = find_input_root(SOURCES)
print("INPUT_ROOT :", INPUT_ROOT)

OUTPUT_ROOT = Path("/kaggle/working/ocr_output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

RENDER_DPI    = 200
OCR_LANG      = "en"
SAVE_BLOCKS   = True
LOG_EVERY     = 25
EXTRACTOR_TAG = f"v2-paddleocr{PADDLEOCR_MAJOR}-ppocrv5"

all_pdfs = []
for src in SOURCES:
    src_dir = INPUT_ROOT / src
    if not src_dir.is_dir():
        print(f"  [!] missing source dir: {src_dir}")
        continue
    pdfs = sorted(src_dir.glob("*.pdf"))
    print(f"  [{src:<16}] {len(pdfs):>4} PDFs")
    all_pdfs.extend((src, p) for p in pdfs)

print(f"\nTOTAL PDFs : {len(all_pdfs)}")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


PaddlePaddle : 3.0.0
PaddleOCR    : 3.7.0
GPUs         : 2
CUDA built   : True
→ Will use PaddleOCR 3.x API
  → matched 4/4 source dirs at /kaggle/input/datasets/fayaz042/ocr-required
INPUT_ROOT : /kaggle/input/datasets/fayaz042/ocr-required
OUTPUT_ROOT: /kaggle/working/ocr_output
  [alerts          ]   45 PDFs
  [fdc             ]   66 PDFs
  [gazette         ]  102 PDFs
  [public_notices  ]  590 PDFs

TOTAL PDFs : 803


In [4]:
# Cell 3 (revised) — PaddleOCR with server det + server rec

import time

print("Initializing PaddleOCR 3.x (PP-OCRv5 server det + server rec) ...")
t0 = time.time()

ocr = PaddleOCR(
    device="gpu:0",
    lang=OCR_LANG,
    ocr_version="PP-OCRv5",
    text_detection_model_name="PP-OCRv5_server_det",     # explicit
    text_recognition_model_name="PP-OCRv5_server_rec",   # server, not mobile
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_rec_score_thresh=0.0,
)

print(f"PaddleOCR ready in {time.time() - t0:.1f}s")
print(f"Device: {paddle.device.get_device()}")

/tmp/ipykernel_224/2687535828.py:8: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  ocr = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


Initializing PaddleOCR 3.x (PP-OCRv5 server det + server rec) ...


Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None, None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

PaddleOCR ready in 4.8s
Device: gpu:0


In [7]:
# Cell 4 (fixed) — Test on 1 PDF with PaddleOCR 3.x v3 API

import io
import time
import numpy as np
import pymupdf
from PIL import Image


def render_page(pdf_page, dpi=RENDER_DPI):
    """Render one PyMuPDF page to a numpy RGB array."""
    pix = pdf_page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    return np.array(img)


def ocr_page_v3(img_np):
    """
    Run PaddleOCR 3.x predict() on a numpy RGB image.
    Returns (text, blocks, error).

    IMPORTANT: OCRResult.json returns {'res': {rec_texts, rec_scores, ...}}.
    The actual payload is nested under the 'res' key.
    """
    try:
        results = ocr.predict(img_np)
    except Exception as exc:
        return "", [], str(exc)[:200]

    if not results:
        return "", [], None

    res = results[0]

    # Unwrap the OCRResult -> dict -> payload
    data = res.json if hasattr(res, "json") else res
    if isinstance(data, dict) and "res" in data:
        data = data["res"]

    rec_texts  = data.get("rec_texts", [])  or []
    rec_scores = data.get("rec_scores", []) or []
    rec_boxes  = data.get("rec_boxes", None)
    rec_polys  = data.get("rec_polys", None)

    lines, blocks = [], []
    for i, text in enumerate(rec_texts):
        text = (text or "").strip()
        if not text:
            continue

        conf = float(rec_scores[i]) if i < len(rec_scores) else 0.0

        # Prefer axis-aligned rect; fall back to 4-point polygon
        if rec_boxes is not None and i < len(rec_boxes):
            x1, y1, x2, y2 = [int(v) for v in rec_boxes[i]]
            bbox = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
        elif rec_polys is not None and i < len(rec_polys):
            bbox = [[int(x), int(y)] for x, y in rec_polys[i]]
        else:
            bbox = []

        lines.append(text)
        blocks.append({
            "bbox": bbox,
            "text": text,
            "confidence": round(conf, 4),
        })

    return "\n".join(lines), blocks, None


def process_pdf(source, pdf_path):
    """Process one PDF -> dict matching the Stage 2 contract."""
    doc_id = f"{source}/{pdf_path.name}"
    pages_out = []
    t0 = time.time()

    try:
        doc = pymupdf.open(str(pdf_path))
    except Exception as exc:
        return {
            "doc_id": doc_id,
            "source_name": source,
            "pages": [],
            "tables": [],
            "extraction_meta": {
                "text_pages": 0, "table_pages": 0, "ocr_pages": 0,
                "empty_pages": 0, "failed_pages": 0, "total_pages": 0,
                "extractor_version": EXTRACTOR_TAG,
                "error": str(exc)[:200],
            },
        }

    try:
        for pno in range(len(doc)):
            try:
                img = render_page(doc[pno], dpi=RENDER_DPI)
                text, blocks, err = ocr_page_v3(img)

                page_out = {
                    "page": pno + 1,
                    "type": "image",
                    "method": "paddleocr-v3",
                    "text": text,
                    "tables": [],
                }
                if SAVE_BLOCKS:
                    page_out["ocr_blocks"] = blocks
                if err:
                    page_out["error"] = err
                pages_out.append(page_out)

            except Exception as exc:
                pages_out.append({
                    "page": pno + 1,
                    "type": "image",
                    "method": "paddleocr-v3-failed",
                    "text": "",
                    "tables": [],
                    "error": str(exc)[:200],
                })
    finally:
        doc.close()

    n_pages = len(pages_out)
    failed  = sum(1 for p in pages_out if p["method"].endswith("failed"))
    empty   = sum(1 for p in pages_out if not p.get("text") and not p.get("error"))

    return {
        "doc_id": doc_id,
        "source_name": source,
        "pages": pages_out,
        "tables": [],
        "extraction_meta": {
            "text_pages": 0,
            "table_pages": 0,
            "ocr_pages": n_pages - failed,
            "empty_pages": empty,
            "failed_pages": failed,
            "total_pages": n_pages,
            "extractor_version": EXTRACTOR_TAG,
            "processing_time_sec": round(time.time() - t0, 2),
        },
    }


# ---- Smoke test on the first PDF ----
src, path = all_pdfs[0]
print(f"TEST: {src}/{path.name}")
t0 = time.time()
result = process_pdf(src, path)
elapsed = time.time() - t0

m = result["extraction_meta"]
print(f"  pages  : {m['total_pages']}")
print(f"  ocr    : {m['ocr_pages']}")
print(f"  empty  : {m['empty_pages']}")
print(f"  failed : {m['failed_pages']}")
print(f"  time   : {elapsed:.2f}s  ({elapsed / max(1, m['total_pages']):.2f}s/page)")

if result["pages"]:
    p = result["pages"][0]
    preview = (p.get("text") or "")[:400].replace("\n", " | ")
    print(f"\n  page 1 preview: {preview}")
    print(f"  page 1 blocks : {len(p.get('ocr_blocks', []))}")
    print(f"  page 1 first block: {p['ocr_blocks'][0] if p.get('ocr_blocks') else 'none'}")

# ---- Per-page timing breakdown (to see warm vs cold) ----
if result["pages"]:
    print(f"\n  Per-page text lengths:")
    for pg in result["pages"]:
        print(f"    page {pg['page']:>2}: {len(pg['text']):>5} chars, "
              f"{len(pg.get('ocr_blocks', [])):>3} blocks")

# ---- Rough full-run estimate ----
per_page = elapsed / max(1, m["total_pages"])
print(f"\n  Estimated total (single GPU, ~5 pages/PDF avg): "
      f"{per_page * 803 * 5 / 60:.0f} min")
print(f"  Estimated total (dual GPU,    ~5 pages/PDF avg): "
      f"{per_page * 803 * 5 / 60 / 2:.0f} min")

TEST: alerts/10182_Medical_Device_Alert_dated_08_May_2023.pdf
  pages  : 12
  ocr    : 12
  empty  : 0
  failed : 0
  time   : 47.94s  (3.99s/page)

  page 1 preview: Central Drugs Standard Control Organization | Directorate General of Health Services | Ministry of Health and Family Welfare | Government of India | (Medical Devices and Diagnostics Division) | Email: ddcimd-cdsco@nic.in | Food & Drugs Administration Bhawan, | Kotla Road, New Delhi. | F.No. 29/Misc./03/2023-DC(134) | Date: 08th May 2023 | MEDICAL DEVICE ALERT | DEVICE | HOYA Vivinex Toric IOLs (Model Numbers: XY1AT2, XY1
  page 1 blocks : 40
  page 1 first block: {'bbox': [[452, 110], [1244, 110], [1244, 154], [452, 154]], 'text': 'Central Drugs Standard Control Organization', 'confidence': 0.9796}

  Per-page text lengths:
    page  1:  1817 chars,  40 blocks
    page  2:  4286 chars, 454 blocks
    page  3:  5355 chars, 598 blocks
    page  4:  5977 chars, 665 blocks
    page  5:  6168 chars, 684 blocks
    page  6:  63

In [8]:
# Cell 5 — Dual-GPU parallel OCR over all PDFs
# Writes a worker script, launches 2 subprocesses (one per T4), streams output.

import os, sys, json, time, subprocess, threading
from pathlib import Path

run_start = time.time()

# ============================================================================
# 1. Worker script (runs as a separate Python process per GPU)
# ============================================================================
WORKER_CODE = r'''
import os, sys, io, json, time, argparse
from pathlib import Path

# --- Patch find_spec BEFORE importing paddleocr (blocks modelscope->torch) ---
import importlib.util
_real_find_spec = importlib.util.find_spec
def _patched_find_spec(name, *a, **k):
    if name == "torch":
        return None
    return _real_find_spec(name, *a, **k)
importlib.util.find_spec = _patched_find_spec

os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import numpy as np
import pymupdf
from PIL import Image
import paddle
from paddleocr import PaddleOCR

# ---------------- CLI ----------------
ap = argparse.ArgumentParser()
ap.add_argument("--worker-id", required=True)
ap.add_argument("--tasks",     required=True)
ap.add_argument("--output-dir", required=True)
ap.add_argument("--lock-dir",   required=True)
ap.add_argument("--dpi", type=int, default=200)
ap.add_argument("--extractor-tag", default="v2-paddleocr3-ppocrv5")
args = ap.parse_args()

WORKER_ID    = args.worker_id
OUTPUT_DIR   = Path(args.output_dir);  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCK_DIR     = Path(args.lock_dir);    LOCK_DIR.mkdir(parents=True, exist_ok=True)
DPI          = args.dpi
EXTRACTOR_TAG = args.extractor_tag

# ---------------- Init OCR ----------------
print(f"[{WORKER_ID}] CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}", flush=True)
t0 = time.time()
ocr = PaddleOCR(
    device="gpu:0",                      # CUDA_VISIBLE_DEVICES already restricts us
    lang="en",
    ocr_version="PP-OCRv5",
    text_detection_model_name="PP-OCRv5_server_det",
    text_recognition_model_name="PP-OCRv5_server_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_rec_score_thresh=0.0,
)
print(f"[{WORKER_ID}] OCR ready in {time.time()-t0:.1f}s on {paddle.device.get_device()}", flush=True)

# ---------------- Core helpers ----------------
def render_page(page, dpi):
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    return np.array(img)

def ocr_page(img_np):
    try:
        results = ocr.predict(img_np)
    except Exception as exc:
        return "", [], str(exc)[:200]
    if not results:
        return "", [], None
    res = results[0]
    data = res.json if hasattr(res, "json") else res
    if isinstance(data, dict) and "res" in data:
        data = data["res"]
    rec_texts  = data.get("rec_texts", [])  or []
    rec_scores = data.get("rec_scores", []) or []
    rec_boxes  = data.get("rec_boxes")
    rec_polys  = data.get("rec_polys")

    lines, blocks = [], []
    for i, text in enumerate(rec_texts):
        text = (text or "").strip()
        if not text: continue
        conf = float(rec_scores[i]) if i < len(rec_scores) else 0.0
        if rec_boxes is not None and i < len(rec_boxes):
            x1, y1, x2, y2 = [int(v) for v in rec_boxes[i]]
            bbox = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
        elif rec_polys is not None and i < len(rec_polys):
            bbox = [[int(x), int(y)] for x, y in rec_polys[i]]
        else:
            bbox = []
        lines.append(text)
        blocks.append({"bbox": bbox, "text": text, "confidence": round(conf, 4)})
    return "\n".join(lines), blocks, None

def process_one(source, pdf_path):
    doc_id = f"{source}/{pdf_path.name}"
    pages_out = []
    t0 = time.time()
    try:
        doc = pymupdf.open(str(pdf_path))
    except Exception as exc:
        return {"doc_id": doc_id, "source_name": source, "pages": [], "tables": [],
                "extraction_meta": {"total_pages": 0, "error": str(exc)[:200],
                                    "extractor_version": EXTRACTOR_TAG}}
    try:
        for pno in range(len(doc)):
            try:
                img = render_page(doc[pno], DPI)
                text, blocks, err = ocr_page(img)
                po = {"page": pno+1, "type": "image", "method": "paddleocr-v3",
                      "text": text, "tables": [], "ocr_blocks": blocks}
                if err: po["error"] = err
                pages_out.append(po)
            except Exception as exc:
                pages_out.append({"page": pno+1, "type": "image",
                                  "method": "paddleocr-v3-failed", "text": "",
                                  "tables": [], "error": str(exc)[:200]})
    finally:
        doc.close()
    n_pages = len(pages_out)
    failed  = sum(1 for p in pages_out if p["method"].endswith("failed"))
    empty   = sum(1 for p in pages_out if not p.get("text") and not p.get("error"))
    return {
        "doc_id": doc_id, "source_name": source, "pages": pages_out, "tables": [],
        "extraction_meta": {
            "text_pages": 0, "table_pages": 0,
            "ocr_pages": n_pages - failed, "empty_pages": empty,
            "failed_pages": failed, "total_pages": n_pages,
            "extractor_version": EXTRACTOR_TAG,
            "processing_time_sec": round(time.time() - t0, 2),
        },
    }

def safe_name(doc_id):
    return doc_id.replace("/", "__").replace("\\", "__")

def try_claim(doc_id, stale_sec=900):
    """Atomically claim a task via O_CREAT|O_EXCL. Reclaims stale locks."""
    out_path = OUTPUT_DIR / f"{safe_name(doc_id)}.json"
    if out_path.exists():
        return False
    lock_path = LOCK_DIR / f"{safe_name(doc_id)}.lock"
    for _ in range(2):
        try:
            fd = os.open(str(lock_path), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, f"{WORKER_ID}:{os.getpid()}\n".encode())
            os.close(fd)
            return True
        except FileExistsError:
            try:
                if time.time() - lock_path.stat().st_mtime > stale_sec:
                    lock_path.unlink(missing_ok=True)
                    continue
            except FileNotFoundError:
                continue
            return False
    return False

# ---------------- Main loop ----------------
tasks = json.loads(Path(args.tasks).read_text())
print(f"[{WORKER_ID}] {len(tasks)} tasks in queue", flush=True)

done = skipped = errors = pages_total = 0
loop_t0 = time.time()
last_log = loop_t0

for task in tasks:
    source   = task["source"]
    pdf_path = Path(task["path"])
    doc_id   = f"{source}/{pdf_path.name}"

    if not try_claim(doc_id):
        skipped += 1
        continue

    try:
        result = process_one(source, pdf_path)
        (OUTPUT_DIR / f"{safe_name(doc_id)}.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        pages_total += result["extraction_meta"].get("total_pages", 0)
        done += 1
    except Exception as exc:
        errors += 1
        print(f"[{WORKER_ID}] ERROR {doc_id}: {exc}", flush=True)

    now = time.time()
    if done % 10 == 0 or (now - last_log) > 30:
        el = now - loop_t0
        rate = done / el if el > 0 else 0
        rem = len(tasks) - done - skipped
        eta = rem / rate if rate > 0 else 0
        print(f"[{WORKER_ID}] done={done} skip={skipped} err={errors} "
              f"pages={pages_total} elapsed={el/60:.1f}m eta={eta/60:.1f}m", flush=True)
        last_log = now

print(f"[{WORKER_ID}] FINISHED done={done} skip={skipped} err={errors} "
      f"pages={pages_total} in {(time.time()-loop_t0)/60:.1f}m", flush=True)
'''

# ============================================================================
# 2. Prepare task list + lock dir
# ============================================================================
TASKS_FILE = Path("/kaggle/working/_ocr_tasks.json")
LOCK_DIR   = Path("/kaggle/working/_ocr_locks")
LOCK_DIR.mkdir(parents=True, exist_ok=True)

tasks_payload = [{"source": s, "path": str(p)} for s, p in all_pdfs]
TASKS_FILE.write_text(json.dumps(tasks_payload), encoding="utf-8")
print(f"Wrote {len(tasks_payload)} tasks to {TASKS_FILE}")

# ============================================================================
# 3. Write worker script
# ============================================================================
WORKER_PATH = Path("/kaggle/working/ocr_worker.py")
WORKER_PATH.write_text(WORKER_CODE, encoding="utf-8")
print(f"Wrote worker script to {WORKER_PATH}")

# ============================================================================
# 4. Launch two subprocesses, one per GPU
# ============================================================================
def make_env(gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
    return env

base_cmd = [
    sys.executable, str(WORKER_PATH),
    "--tasks",     str(TASKS_FILE),
    "--output-dir", str(OUTPUT_ROOT),
    "--lock-dir",   str(LOCK_DIR),
    "--dpi",        str(RENDER_DPI),
    "--extractor-tag", EXTRACTOR_TAG,
]

print("\nLaunching 2 workers ...")
p0 = subprocess.Popen(base_cmd + ["--worker-id", "W0"], env=make_env(0),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1)
p1 = subprocess.Popen(base_cmd + ["--worker-id", "W1"], env=make_env(1),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1)
print(f"  W0 pid={p0.pid} on GPU 0")
print(f"  W1 pid={p1.pid} on GPU 1")
print("Streaming worker logs (Ctrl+C the cell to abort)...\n")

def stream(pipe, prefix):
    try:
        for line in iter(pipe.readline, ""):
            print(f"[{prefix}] {line.rstrip()}", flush=True)
    finally:
        pipe.close()

t0 = threading.Thread(target=stream, args=(p0.stdout, "W0"), daemon=True)
t1 = threading.Thread(target=stream, args=(p1.stdout, "W1"), daemon=True)
t0.start(); t1.start()

try:
    p0.wait(); p1.wait()
finally:
    for p in (p0, p1):
        if p.poll() is None:
            p.terminate()
            try:    p.wait(timeout=10)
            except subprocess.TimeoutExpired: p.kill()

print(f"\nW0 exit code: {p0.returncode}")
print(f"W1 exit code: {p1.returncode}")

# ============================================================================
# 5. Summary
# ============================================================================
out_files = sorted(OUTPUT_ROOT.glob("*.json"))
total_pages = total_failed = total_empty = 0
for f in out_files:
    try:
        d = json.loads(f.read_text(encoding="utf-8"))
        m = d.get("extraction_meta", {})
        total_pages  += m.get("total_pages", 0)
        total_failed += m.get("failed_pages", 0)
        total_empty  += m.get("empty_pages", 0)
    except Exception:
        pass

runtime_min = (time.time() - run_start) / 60
print(f"\n=== SUMMARY ===")
print(f"PDFs processed : {len(out_files)} / {len(all_pdfs)}")
print(f"Total pages    : {total_pages}")
print(f"Failed pages   : {total_failed}")
print(f"Empty pages    : {total_empty}")
print(f"Runtime        : {runtime_min:.1f} min")

summary = {
    "total_docs":   len(all_pdfs),
    "processed":    len(out_files),
    "total_pages":  total_pages,
    "failed_pages": total_failed,
    "empty_pages":  total_empty,
    "runtime_min":  round(runtime_min, 1),
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
Path("/kaggle/working/ocr_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8")
print("Wrote /kaggle/working/ocr_summary.json")

Wrote 803 tasks to /kaggle/working/_ocr_tasks.json
Wrote worker script to /kaggle/working/ocr_worker.py

Launching 2 workers ...
  W0 pid=15795 on GPU 0
  W1 pid=15796 on GPU 1
Streaming worker logs (Ctrl+C the cell to abort)...

[W1] /usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
[W1]   warnings.warn(warning_message)
[W0] /usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
[W0]   warnings.warn(warning_message)
[W1] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data u

In [9]:
# Cell 6 — Audit empty pages + write CSV for downstream use

import json
import csv
from pathlib import Path
from collections import Counter, defaultdict

out_files = sorted(OUTPUT_ROOT.glob("*.json"))
print(f"Total doc JSONs: {len(out_files)}\n")

# ---- Aggregate stats per source ----
src_stats = defaultdict(lambda: {
    "docs": 0, "pages": 0, "empty": 0, "failed": 0,
    "total_chars": 0, "total_blocks": 0,
})
empty_pages_index = []  # (source, doc_id, page_no, n_chars, n_blocks)
low_conf_blocks = 0
total_blocks = 0

for f in out_files:
    try:
        d = json.loads(f.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[parse-error] {f.name}: {e}")
        continue

    src = d.get("source_name", "?")
    src_stats[src]["docs"] += 1
    m = d.get("extraction_meta", {})
    src_stats[src]["pages"]  += m.get("total_pages", 0)
    src_stats[src]["empty"]  += m.get("empty_pages", 0)
    src_stats[src]["failed"] += m.get("failed_pages", 0)

    for p in d.get("pages", []):
        text = p.get("text") or ""
        blocks = p.get("ocr_blocks", [])
        src_stats[src]["total_chars"]  += len(text)
        src_stats[src]["total_blocks"] += len(blocks)
        total_blocks += len(blocks)
        for b in blocks:
            if b.get("confidence", 1.0) < 0.6:
                low_conf_blocks += 1
        if not text.strip() and not p.get("error"):
            empty_pages_index.append((
                src, d["doc_id"], p["page"], len(text), len(blocks),
            ))

print("=== Per-source stats ===")
print(f"{'source':<16} {'docs':>5} {'pages':>6} {'empty':>6} {'failed':>6} "
      f"{'chars':>10} {'blocks':>8} {'chars/page':>10}")
for src, s in sorted(src_stats.items()):
    cpp = s["total_chars"] / max(1, s["pages"])
    print(f"{src:<16} {s['docs']:>5} {s['pages']:>6} {s['empty']:>6} "
          f"{s['failed']:>6} {s['total_chars']:>10,} {s['total_blocks']:>8,} "
          f"{cpp:>10.1f}")

print(f"\nLow-confidence blocks (<0.6): {low_conf_blocks:,} / {total_blocks:,} "
      f"({100*low_conf_blocks/max(1,total_blocks):.2f}%)")

# ---- Which docs have empty pages? ----
empty_by_doc = Counter(doc_id for _, doc_id, _, _, _ in empty_pages_index)
print(f"\nEmpty pages found in {len(empty_by_doc)} distinct docs")

if empty_by_doc:
    print("\nTop 10 docs by empty-page count:")
    for doc_id, n in empty_by_doc.most_common(10):
        print(f"  {n:>3} empty  {doc_id}")

# ---- Save empty-page audit ----
audit_path = Path("/kaggle/working/empty_pages_audit.csv")
with audit_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["source", "doc_id", "page", "n_chars", "n_blocks"])
    w.writerows(empty_pages_index)
print(f"\nWrote {audit_path}  ({len(empty_pages_index)} rows)")

# ---- Flat CSV for downstream (one row per page) ----
flat_path = Path("/kaggle/working/ocr_pages_flat.csv")
with flat_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["doc_id", "source", "page", "n_chars", "n_blocks",
                "mean_conf", "text"])
    for f in out_files:
        try:
            d = json.loads(f.read_text(encoding="utf-8"))
        except Exception:
            continue
        for p in d.get("pages", []):
            blocks = p.get("ocr_blocks", [])
            confs = [b["confidence"] for b in blocks if "confidence" in b]
            w.writerow([
                d["doc_id"], d.get("source_name", "?"), p["page"],
                len(p.get("text") or ""), len(blocks),
                round(sum(confs)/len(confs), 4) if confs else "",
                (p.get("text") or "").replace("\r", " "),
            ])
print(f"Wrote {flat_path}  (flat page-level CSV for downstream use)")

Total doc JSONs: 803

=== Per-source stats ===
source            docs  pages  empty failed      chars   blocks chars/page
alerts              45    438     18      0    853,935   39,150     1949.6
fdc                 66    471     12      0  1,296,638   43,011     2752.9
gazette            102    453      3      0    560,360   20,584     1237.0
public_notices     590   4119     48      0  7,808,670  254,533     1895.8

Low-confidence blocks (<0.6): 10,433 / 357,278 (2.92%)

Empty pages found in 22 distinct docs

Top 10 docs by empty-page count:
   17 empty  alerts/3257_Report_on_countrywide_survey_for_Spurious_Drugs.pdf
   17 empty  public_notices/13342_ICMR-CDSCO_Standard_Performance_Evaluation_Protocols.pdf
   10 empty  fdc/845_Public_Notice_regarding_Meeting_of_DTAB_sub_Committee_w.ef_05.06.2018_to_22.06.2018_in_the_office_of_DCGI_to_examine_the.pdf
   10 empty  public_notices/845_Public_Notice_regarding_Meeting_of_DTAB_sub_Committee_w.ef_05.06.2018_to_22.06.2018_in_the_office_of_DC

In [10]:
# Cell 7 — Zip ONLY the per-doc JSON files from ocr_output/

import shutil, os, json
from pathlib import Path

# Source = the folder with per-doc JSONs
SRC = OUTPUT_ROOT                       # /kaggle/working/ocr_output

# Destination inside /kaggle/working/
ZIP_BASE = Path("/kaggle/working/ocr_output_jsons")

# Sanity: confirm it's the right folder
jsons = sorted(SRC.glob("*.json"))
print(f"Source folder : {SRC}")
print(f"JSON count    : {len(jsons)}")
if jsons:
    print(f"Sample names  :")
    for f in jsons[:5]:
        print(f"  {f.name}")

assert len(jsons) > 0, "No JSONs found — check OUTPUT_ROOT"

# Build zip: base_name.zip in /kaggle/working/
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", SRC)
size_mb = os.path.getsize(zip_path) / 1024 / 1024

print(f"\n✅ Zip created : {zip_path}")
print(f"   Size        : {size_mb:.1f} MB")
print(f"   Contains    : {len(jsons)} JSON files")

# Show what's in the zip (first 10 entries)
import zipfile
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    print(f"\n   Entries in zip: {len(names)}")
    for n in names[:10]:
        info = zf.getinfo(n)
        print(f"     {n}  ({info.file_size/1024:.1f} KB)")
    if len(names) > 10:
        print(f"     ... and {len(names) - 10} more")

# ---- Download instructions ----
print("\n=== Kaggle download ===")
print("Right sidebar → Output → 'ocr_output_jsons.zip' → Download")
print("(Or from the Data tab after the notebook finishes.)")

# ---- Optional: also keep a small manifest ----
manifest = {
    "n_jsons":   len(jsons),
    "zip_path":  zip_path,
    "size_mb":   round(size_mb, 2),
    "created_at": __import__("time").strftime("%Y-%m-%d %H:%M:%S"),
    "files":     [f.name for f in jsons],
}
Path("/kaggle/working/ocr_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8")
print("\nAlso wrote /kaggle/working/ocr_manifest.json")

Source folder : /kaggle/working/ocr_output
JSON count    : 803
Sample names  :
  alerts__10182_Medical_Device_Alert_dated_08_May_2023.pdf.json
  alerts__10354_Alert_on_Pholcodine_containing_cough_and_cold_remedies.pdf.json
  alerts__10417_Alert_on_Suspected_Faisified_Versions_of_Glucagon-Like_Peptide_1_Receptor_Agonist_GLP_1_RA_Products.pdf.json
  alerts__10494_Voluntary_Recall_of_Digene_Gel_manufactured_by_Ms_Abbott_India_Limited_L-1819_Verna_Industrial_Estate_Salcette_Goa-40372.pdf.json
  alerts__10504_Alert_on_falsified_versions_of_Adcetris_Injection_Brentuximab_Vedotin_Manufactured_by_Ms_Takeda_Pharmaceuticals_Company_.pdf.json

✅ Zip created : /kaggle/working/ocr_output_jsons.zip
   Size        : 14.5 MB
   Contains    : 803 JSON files

   Entries in zip: 803
     public_notices__2192_Presentation_for_Rabies_Vaccine.pdf.json  (12.8 KB)
     public_notices__2422_Disposal_of_unserviceableobsolete_item_of_CDSCO_dated_21.12.2018.pdf.json  (33.3 KB)
     fdc__10778_To_update_package_in